# SPiCe Tutorial: Modelling Epithelial-Mesenchymal Plasticity

This notebook demonstrates the full SPiCe workflow for breast cancer Xenium
spatial transcriptomics data using the `spice.tl` / `spice.pl` API.

All tool functions modify `adata` in place — results are stored in
`adata.obs`, `adata.obsm` and `adata.uns['spice']`.

In [1]:
import os
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import numpy as np
import pandas as pd
import scanpy as sc

In [4]:
sc.set_figure_params(frameon=False)

In [5]:
project_dir = "/home/cenk/Withnell_spice/"
os.chdir(project_dir)

figures_dir = project_dir + "figures/"

## 1. Load data into AnnData

In [6]:
adata = sc.read_h5ad("processed_data/xenium_lung_hallmark_spice_results.h5ad")

In [7]:
adata.obs["array_row"] = adata.obsm["spatial"][:, 0]
adata.obs["array_col"] = adata.obsm["spatial"][:, 1]

In [8]:
adata.obs["cell_type"].value_counts()

cell_type
T cells                                 45830
Fibroblasts                             36728
Endothelial cells                       30419
Macrophages                             25832
Epithelial tumour cells                 21976
Pericytes                               18922
Endothelial cells (activated/plvap+)    12995
Ciliated epithelial cells               12730
Plasma cells                             9681
Alveolar type I epithelial cells         7809
Alveolar type II epithelial cells        7580
Regulatory T cells                       5646
Low-quality / unassigned                 3328
NK cells                                  682
Name: count, dtype: int64

In [9]:
def calculateDistancesHelper(
    adata, primary_variables, comparison_variables, hotspot_number_bool=False, k=10
):

    coords = adata.obs[["array_row", "array_col"]].to_numpy()

    primary_masks = _standardise_cell_masks(adata, primary_variables)
    comparison_masks = _standardise_cell_masks(adata, comparison_variables)

    results = []

    for p_var, p_mask in primary_masks.items():
        p_idx = np.where(p_mask)[0]
        if len(p_idx) == 0:
            continue

        p_coords = coords[p_idx]

        if hotspot_number_bool:
            hotspot_col = f"{p_var}_number"
            hotspot_labels = adata.obs.loc[p_mask, hotspot_col].to_numpy()

        for c_var, c_mask in comparison_masks.items():
            c_idx = np.where(c_mask)[0]
            if len(c_idx) == 0:
                continue

            c_coords = coords[c_idx]

            nn = NearestNeighbors(n_neighbors=min(k, len(c_coords)), metric="euclidean")
            nn.fit(c_coords)

            distances, _ = nn.kneighbors(p_coords)

            mean_dist = distances.mean(axis=1)

            df = pd.DataFrame(
                {
                    "mean_knn_distance": mean_dist,
                    "primary_variable": p_var,
                    "comparison_variable": c_var,
                    "primary_index": adata.obs.index[p_idx],
                }
            )

            if hotspot_number_bool:
                df["hotspot_number"] = hotspot_labels

            results.append(df)

    return pd.concat(results, ignore_index=True)


import numpy as np
import pandas as pd
import squidpy as sq
from scipy.sparse.csgraph import connected_components
from sklearn.neighbors import NearestNeighbors


def _standardise_cell_masks(adata, variables):
    """
    Converts variable names into boolean masks over cells.

    Supports:
    1. One-hot columns in adata.obs
    2. Single categorical column (cell_type, annotation, etc.)
    """

    # detect categorical column
    categorical_col = None
    for col in ["cell_type", "annotation", "celltype", "cluster"]:
        if col in adata.obs.columns:
            categorical_col = col
            break

    masks = {}

    for var in variables:
        if var in adata.obs.columns:
            masks[var] = adata.obs[var].notnull().to_numpy()

        elif categorical_col is not None:
            masks[var] = adata.obs[categorical_col].to_numpy() == var

        else:
            raise KeyError(
                f"Variable '{var}' not found in obs columns "
                f"and no categorical annotation column detected."
            )

    return masks


def calculateDistances(
    anndata,
    primary_variables,
    comparison_variables=None,
    split_by_slide_in_batch=True,
    hotspot_number=False,
    k=10,
):

    if comparison_variables is None:
        comparison_variables = primary_variables

    if isinstance(primary_variables, str):
        primary_variables = [primary_variables]

    if isinstance(comparison_variables, str):
        comparison_variables = [comparison_variables]

    adata = anndata.copy()
    results = []

    if split_by_slide_in_batch:
        sq.gr.spatial_neighbors(adata, n_rings=1, coord_type="grid", n_neighs=6)

        conn = adata.obsp["spatial_connectivities"]

        _, labels = connected_components(conn, directed=False, return_labels=True)

        adata.obs["connected_labels"] = labels

        for label in np.unique(labels):
            sub = adata[adata.obs["connected_labels"] == label].copy()

            if sub.n_obs < 30:
                continue

            results.append(
                calculateDistancesHelper(
                    sub,
                    primary_variables,
                    comparison_variables,
                    hotspot_number_bool=hotspot_number,
                    k=k,
                )
            )

    else:
        results.append(
            calculateDistancesHelper(
                adata,
                primary_variables,
                comparison_variables,
                hotspot_number_bool=hotspot_number,
                k=k,
            )
        )

    return pd.concat(results, ignore_index=True)

In [10]:
cell_types = [ct for ct in adata.obs["cell_type"] if ct != "Epithelial tumour cells"]

In [11]:
distances = calculateDistances(
    anndata=adata,
    primary_variables=["Epithelial tumour cells"],
    comparison_variables=cell_types,
    split_by_slide_in_batch=False,
    hotspot_number=False,
    k=10,
)

In [12]:
wide = distances.pivot(
    index="primary_index", columns="comparison_variable", values="mean_knn_distance"
)

wide.columns = [f"min_dist_to_{c}" for c in wide.columns]

# make sure indices are strings and aligned
wide.index = wide.index.astype(str)
adata.obs.index = adata.obs.index.astype(str)

# add to obs
adata.obs = adata.obs.join(wide, how="left")

In [13]:
# save Xenium metadata with distances
metadata = adata.obs
metadata.to_csv("results/xenium_lung_5k_metadata_with_distances.csv", index=False)